# Nested Leave-One-Device-Out Validation with Fold-Specific Top-20 Features

## 1. Experiment overview

This notebook performs strict, fold-isolated Leave-One-Device-Out (LODO) evaluation across nine IoT devices. In every fold:

1. one complete device is held out;
2. a 115-feature baseline Random Forest is fitted using only the remaining eight devices;
3. that fold's Top-20 features are selected from the baseline model's feature importances;
4. a fresh Random Forest is fitted on the same eight devices using only those fold-specific Top-20 features;
5. the model is evaluated on the held-out device.

The held-out device contributes no rows or labels to either feature selection or classifier training. Schema inspection and descriptive sample counts do not affect either fitted model. No global feature-importance file and no existing `data/lodo_top20/` cache is used.

Every fold's full feature-importance table, ranked Top-20 list, performance metrics, confusion matrix, and classification report are written immediately to a new output namespace. Existing LODO results are not overwritten.

## 2. Imports and configuration

Both Random Forest fits use the exact current project configuration, including the source scripts' explicit logging parameter `verbose=1`. All other estimator parameters remain at scikit-learn defaults. Scaling, SMOTE, and class weighting are not introduced.

In [ ]:
from __future__ import annotations

import gc
import json
import platform
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import imblearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "workflow.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Start Jupyter from the project or notebooks directory."
    )


PROJECT_ROOT = locate_project_root()
LABELED_DEVICE_DIR = PROJECT_ROOT / "data" / "labeled_devices"
DEVICE_INFO_PATH = PROJECT_ROOT / "archive-2" / "device_info.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "reports" / "lodo_nested_top20"
FEATURE_IMPORTANCE_DIR = OUTPUT_ROOT / "feature_importance_115"
TOP20_LIST_DIR = OUTPUT_ROOT / "top20_lists"
PERFORMANCE_DIR = OUTPUT_ROOT / "performance"
CONFUSION_DIR = OUTPUT_ROOT / "confusion_matrices"
CLASSIFICATION_REPORT_DIR = OUTPUT_ROOT / "classification_reports"
ALL_RESULTS_PATH = OUTPUT_ROOT / "all_folds_results.csv"
ALL_TOP20_PATH = OUTPUT_ROOT / "all_folds_top20.csv"
RUN_CONFIGURATION_PATH = OUTPUT_ROOT / "run_configuration.json"

TARGET_COLUMN = "binary_target"
DROP_COLUMNS = ["binary_label", "binary_target", "source_file"]
CLASS_LABELS = [0, 1]
CLASS_NAMES = ["benign", "attack"]
TOP_N = 20
CHUNK_SIZE = 100_000
RANDOM_STATE = 42
RF_CONFIG = {
    "n_estimators": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": 1,
}

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Nested LODO output root: {OUTPUT_ROOT.relative_to(PROJECT_ROOT)}")
print(f"Selector and final-model configuration: {RF_CONFIG}")

## 3. Environment information

In [ ]:
def environment_information() -> pd.Series:
    info = {
        "Operating system": platform.system(),
        "OS version": platform.mac_ver()[0] if platform.system() == "Darwin" else platform.release(),
        "Architecture": platform.machine(),
        "Processor": platform.processor() or "not reported",
        "Physical CPU cores": psutil.cpu_count(logical=False),
        "Logical CPU cores": psutil.cpu_count(logical=True),
        "RAM (GiB)": round(psutil.virtual_memory().total / (1024**3), 2),
        "Python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "imbalanced-learn": imblearn.__version__,
    }
    if platform.system() == "Darwin":
        try:
            completed = subprocess.run(
                ["system_profiler", "SPHardwareDataType", "-json"],
                check=True,
                capture_output=True,
                text=True,
            )
            hardware = json.loads(completed.stdout)["SPHardwareDataType"][0]
            info["Computer model"] = hardware.get("machine_name", "not reported")
            info["Model identifier"] = hardware.get("machine_model", "not reported")
            info["Chip"] = hardware.get("chip_type", "not reported")
        except (OSError, subprocess.SubprocessError, KeyError, IndexError, json.JSONDecodeError):
            info["Computer model"] = "not available from system_profiler"
            info["Chip"] = "not available from system_profiler"
    return pd.Series(info, name="Value")


environment_info = environment_information()
display(environment_info.to_frame())

## 4. Canonical full-feature inputs and schema validation

Fold-specific selection requires all 115 features, so the canonical inputs are `data/labeled_devices/device_1_labeled.csv` through `device_9_labeled.csv`. The old Top-20 caches cannot be used because they were produced from one global selector.

Only headers are inspected here. All nine devices must expose exactly the same ordered 115-feature schema followed by `binary_label`, `binary_target`, and `source_file` metadata.

In [ ]:
if not DEVICE_INFO_PATH.is_file():
    raise FileNotFoundError(f"Missing device metadata: {DEVICE_INFO_PATH}")

device_info_frame = pd.read_csv(DEVICE_INFO_PATH)
if device_info_frame["DeviceID"].duplicated().any():
    raise ValueError("device_info.csv contains duplicate DeviceID values.")

device_names = dict(
    zip(
        device_info_frame["DeviceID"].astype(int),
        device_info_frame["DeviceName"],
        strict=True,
    )
)
device_ids = sorted(device_names)
if device_ids != list(range(1, 10)):
    raise ValueError(f"Expected device IDs 1..9, found {device_ids}")

device_paths = {
    device_id: LABELED_DEVICE_DIR / f"device_{device_id}_labeled.csv"
    for device_id in device_ids
}
missing_paths = [path for path in device_paths.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Missing labeled device inputs: {missing_paths}")

device_headers = {
    device_id: pd.read_csv(path, nrows=0).columns.tolist()
    for device_id, path in device_paths.items()
}
canonical_header = device_headers[device_ids[0]]
if len(canonical_header) != len(set(canonical_header)):
    raise ValueError("Duplicate columns found in the canonical device header.")
if not all(header == canonical_header for header in device_headers.values()):
    raise ValueError("Feature/metadata headers or column order differ between devices.")

canonical_feature_columns = [
    column for column in canonical_header if column not in DROP_COLUMNS
]
if len(canonical_feature_columns) != 115:
    raise ValueError(
        f"Expected 115 baseline features, found {len(canonical_feature_columns)}"
    )
if any(column not in canonical_header for column in DROP_COLUMNS):
    raise ValueError(f"Required metadata columns are missing: {DROP_COLUMNS}")

header_validation = pd.DataFrame(
    [
        {
            "Device ID": device_id,
            "Device": device_names[device_id],
            "Total columns": len(device_headers[device_id]),
            "Feature columns": len(
                [column for column in device_headers[device_id] if column not in DROP_COLUMNS]
            ),
            "Header/order matches canonical": device_headers[device_id] == canonical_header,
        }
        for device_id in device_ids
    ]
)
display(header_validation)

## 5. Compact device inspection

This pass reads only `binary_label` and `binary_target` in chunks. It verifies the established mapping (`benign=0`, `attack=1`) and supplies exact row counts for memory-map allocation. These descriptive counts are never passed to a selector or classifier.

In [ ]:
def inspect_device_labels(device_id: int, path: Path) -> dict[str, object]:
    counts = np.zeros(2, dtype="int64")
    rows = 0
    for chunk in pd.read_csv(
        path,
        usecols=["binary_label", TARGET_COLUMN],
        dtype={"binary_label": "category", TARGET_COLUMN: "uint8"},
        chunksize=CHUNK_SIZE,
    ):
        mapped = chunk["binary_label"].map({"benign": 0, "attack": 1})
        if mapped.isna().any() or not np.array_equal(
            mapped.to_numpy(dtype="uint8"),
            chunk[TARGET_COLUMN].to_numpy(dtype="uint8", copy=False),
        ):
            raise ValueError(f"Label/target mapping mismatch in {path}")
        target_values = chunk[TARGET_COLUMN].to_numpy(dtype="uint8", copy=False)
        if not set(np.unique(target_values)).issubset({0, 1}):
            raise ValueError(f"Unexpected target value in {path}")
        counts += np.bincount(target_values, minlength=2)
        rows += len(chunk)

    return {
        "device_id": device_id,
        "device": device_names[device_id],
        "rows": rows,
        "benign": int(counts[0]),
        "attack": int(counts[1]),
        "benign_percent": counts[0] / rows * 100,
        "attack_percent": counts[1] / rows * 100,
    }


device_summary = pd.DataFrame(
    [inspect_device_labels(device_id, device_paths[device_id]) for device_id in device_ids]
).sort_values("device_id").reset_index(drop=True)
device_row_counts = dict(
    zip(device_summary["device_id"], device_summary["rows"], strict=True)
)
display(device_summary)
print(f"Total samples across nine devices: {device_summary['rows'].sum():,}")

## 6. Memory-conscious loading and evaluation utilities

A 6–7 million row, 115-feature `float32` matrix is about 3 GiB before Random Forest workspaces. To avoid retaining nine full DataFrames or creating large concatenation copies, each fold is streamed into temporary disk-backed NumPy memory maps. Folds run sequentially.

After the selector is fitted, its 20 selected columns are projected into a smaller memory map in canonical source-column order. The ranked list is saved in importance order, but final model input order follows the original 115-feature header, matching how the existing pandas `usecols` workflow orders model columns.

In [ ]:
def load_devices_to_memmap(
    selected_device_ids: list[int],
    feature_columns: list[str],
    expected_rows: int,
    temporary_directory: Path,
    stem: str,
) -> tuple[np.memmap, np.memmap, Path, Path, float]:
    x_path = temporary_directory / f"{stem}_features.float32.mmap"
    y_path = temporary_directory / f"{stem}_target.uint8.mmap"
    X = np.memmap(
        x_path,
        mode="w+",
        dtype="float32",
        shape=(expected_rows, len(feature_columns)),
        order="C",
    )
    y = np.memmap(
        y_path,
        mode="w+",
        dtype="uint8",
        shape=(expected_rows,),
    )

    dtypes = {column: "float32" for column in feature_columns}
    dtypes[TARGET_COLUMN] = "uint8"
    offset = 0
    start = perf_counter()

    for device_id in selected_device_ids:
        for chunk in pd.read_csv(
            device_paths[device_id],
            usecols=[*feature_columns, TARGET_COLUMN],
            dtype=dtypes,
            chunksize=CHUNK_SIZE,
        ):
            target = chunk.pop(TARGET_COLUMN)
            if chunk.columns.tolist() != feature_columns:
                raise ValueError(
                    f"Unexpected feature order while loading device {device_id}"
                )
            stop = offset + len(chunk)
            if stop > expected_rows:
                raise ValueError("Loaded more rows than the preallocated matrix allows.")
            X[offset:stop] = chunk.to_numpy(dtype="float32", copy=False)
            y[offset:stop] = target.to_numpy(dtype="uint8", copy=False)
            offset = stop

    if offset != expected_rows:
        raise ValueError(f"Expected {expected_rows:,} rows, loaded {offset:,}")
    X.flush()
    y.flush()
    return X, y, x_path, y_path, perf_counter() - start


def project_feature_memmap(
    X_full: np.memmap,
    selected_column_indices: list[int],
    temporary_directory: Path,
    stem: str,
) -> tuple[np.memmap, Path, float]:
    output_path = temporary_directory / f"{stem}_features.float32.mmap"
    X_selected = np.memmap(
        output_path,
        mode="w+",
        dtype="float32",
        shape=(X_full.shape[0], len(selected_column_indices)),
        order="C",
    )
    start = perf_counter()
    for row_start in range(0, X_full.shape[0], CHUNK_SIZE):
        row_stop = min(row_start + CHUNK_SIZE, X_full.shape[0])
        X_selected[row_start:row_stop] = X_full[
            row_start:row_stop, selected_column_indices
        ]
    X_selected.flush()
    return X_selected, output_path, perf_counter() - start


def evaluate_binary_predictions(
    y_true: np.ndarray, y_pred: np.ndarray
) -> tuple[dict[str, float], np.ndarray, str]:
    attack_precision, attack_recall, attack_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    benign_precision, benign_recall, benign_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=0,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "attack_precision": float(attack_precision),
        "attack_recall": float(attack_recall),
        "attack_f1": float(attack_f1),
        "benign_precision": float(benign_precision),
        "benign_recall": float(benign_recall),
        "benign_f1": float(benign_f1),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    matrix = confusion_matrix(y_true, y_pred, labels=CLASS_LABELS)
    report = classification_report(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
    return metrics, matrix, report

In [ ]:
def atomic_write_csv(frame: pd.DataFrame, path: Path, *, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f"{path.name}.tmp")
    frame.to_csv(temporary_path, index=index)
    temporary_path.replace(path)


def atomic_write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f"{path.name}.tmp")
    temporary_path.write_text(text)
    temporary_path.replace(path)


def fold_output_paths(held_out_device_id: int) -> dict[str, Path]:
    prefix = f"device_{held_out_device_id}"
    return {
        "feature_importance": FEATURE_IMPORTANCE_DIR / f"{prefix}_feature_importance_115.csv",
        "top20": TOP20_LIST_DIR / f"{prefix}_top20.csv",
        "metrics": PERFORMANCE_DIR / f"{prefix}_metrics.csv",
        "confusion_matrix": CONFUSION_DIR / f"{prefix}_confusion_matrix.csv",
        "classification_report": (
            CLASSIFICATION_REPORT_DIR / f"{prefix}_classification_report.txt"
        ),
    }


def relative_output_path(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT))

## 7. Nested LODO evaluation loop

This is the expensive cell. It performs **18 Random Forest fits**: one 115-feature selector and one fold-specific Top-20 classifier for each held-out device. Run folds sequentially on the 16 GB machine; do not parallelize notebook folds on top of `n_jobs=-1`.

Per-fold artifacts are written as soon as they become available, so completed folds remain inspectable if a later fold is interrupted. Rerunning the cell recomputes the experiment and replaces only files inside `outputs/reports/lodo_nested_top20/`; legacy LODO outputs remain untouched.

In [ ]:
output_directories = [
    OUTPUT_ROOT,
    FEATURE_IMPORTANCE_DIR,
    TOP20_LIST_DIR,
    PERFORMANCE_DIR,
    CONFUSION_DIR,
    CLASSIFICATION_REPORT_DIR,
]
for directory in output_directories:
    directory.mkdir(parents=True, exist_ok=True)

run_configuration = {
    "method": "Nested LODO with fold-specific Random Forest feature importance",
    "devices": device_ids,
    "baseline_feature_count": len(canonical_feature_columns),
    "selected_feature_count": TOP_N,
    "target": TARGET_COLUMN,
    "label_mapping": {"benign": 0, "attack": 1},
    "selector_model": "RandomForestClassifier",
    "final_model": "RandomForestClassifier",
    "random_forest_config": RF_CONFIG,
    "scaling": "none",
    "balancing": "none",
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
}
atomic_write_text(
    json.dumps(run_configuration, indent=2),
    RUN_CONFIGURATION_PATH,
)

nested_lodo_results: list[dict[str, object]] = []
all_fold_top20_rows: list[dict[str, object]] = []
nested_lodo_confusion_matrices: dict[int, np.ndarray] = {}

summary_by_device = device_summary.set_index("device_id")

for fold_number, held_out_device_id in enumerate(device_ids, start=1):
    train_device_ids = [
        device_id for device_id in device_ids if device_id != held_out_device_id
    ]
    assert len(train_device_ids) == 8
    assert held_out_device_id not in train_device_ids

    train_rows = int(sum(device_row_counts[device_id] for device_id in train_device_ids))
    test_rows = int(device_row_counts[held_out_device_id])
    train_device_string = " ".join(str(device_id) for device_id in train_device_ids)
    paths = fold_output_paths(held_out_device_id)

    print()
    print(
        f"Nested LODO {fold_number}/{len(device_ids)} - "
        f"held out device_{held_out_device_id} ({device_names[held_out_device_id]})"
    )
    print(f"  Selector/classifier train devices: {train_device_ids}")

    with TemporaryDirectory(prefix=f"nested_lodo_device_{held_out_device_id}_") as temporary:
        temporary_directory = Path(temporary)

        # Feature selection: load and fit using the eight training devices only.
        X_full, y_train, x_full_path, y_train_path, selector_load_seconds = (
            load_devices_to_memmap(
                train_device_ids,
                canonical_feature_columns,
                train_rows,
                temporary_directory,
                "selector_train_115",
            )
        )
        assert X_full.shape == (train_rows, 115)

        selector_model = RandomForestClassifier(**RF_CONFIG)
        selector_fit_start = perf_counter()
        selector_model.fit(X_full, y_train)
        selector_fit_seconds = perf_counter() - selector_fit_start

        fold_importance = (
            pd.DataFrame(
                {
                    "feature": canonical_feature_columns,
                    "importance": selector_model.feature_importances_,
                    "source_column_index": range(len(canonical_feature_columns)),
                }
            )
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )
        fold_importance.insert(0, "rank", np.arange(1, len(fold_importance) + 1))
        ranked_top20 = fold_importance.head(TOP_N)["feature"].tolist()
        if len(ranked_top20) != TOP_N or len(set(ranked_top20)) != TOP_N:
            raise ValueError("Fold-specific Top-20 list is not 20 unique features.")

        selected_feature_set = set(ranked_top20)
        model_feature_columns = [
            feature
            for feature in canonical_feature_columns
            if feature in selected_feature_set
        ]
        if len(model_feature_columns) != TOP_N:
            raise ValueError("Could not map ranked Top-20 into canonical model-column order.")
        model_position = {
            feature: position for position, feature in enumerate(model_feature_columns)
        }

        fold_importance.insert(0, "held_out_device_name", device_names[held_out_device_id])
        fold_importance.insert(0, "held_out_device_id", held_out_device_id)
        fold_importance["selector_train_devices"] = train_device_string
        fold_importance["selected_top20"] = fold_importance["rank"] <= TOP_N

        fold_top20 = fold_importance.head(TOP_N).copy()
        fold_top20["model_column_position"] = (
            fold_top20["feature"].map(model_position).astype(int)
        )
        atomic_write_csv(fold_importance, paths["feature_importance"])
        atomic_write_csv(fold_top20, paths["top20"])

        del selector_model
        gc.collect()

        # Project the exact same eight-device rows into canonical Top-20 model order.
        selected_indices = [
            canonical_feature_columns.index(feature)
            for feature in model_feature_columns
        ]
        X_train_top20, x_top20_path, projection_seconds = project_feature_memmap(
            X_full,
            selected_indices,
            temporary_directory,
            "classifier_train_top20",
        )
        assert X_train_top20.shape == (train_rows, TOP_N)

        del X_full
        gc.collect()
        x_full_path.unlink(missing_ok=True)

        final_model = RandomForestClassifier(**RF_CONFIG)
        final_fit_start = perf_counter()
        final_model.fit(X_train_top20, y_train)
        final_fit_seconds = perf_counter() - final_fit_start

        # The held-out device values are first loaded after feature selection is complete.
        X_test, y_test, x_test_path, y_test_path, test_load_seconds = (
            load_devices_to_memmap(
                [held_out_device_id],
                model_feature_columns,
                test_rows,
                temporary_directory,
                "held_out_test_top20",
            )
        )
        assert X_test.shape == (test_rows, TOP_N)

        predict_start = perf_counter()
        predictions = final_model.predict(X_test)
        predict_seconds = perf_counter() - predict_start

        metrics, matrix, report = evaluate_binary_predictions(y_test, predictions)
        tn, fp, fn, tp = matrix.ravel()
        train_counts = np.bincount(np.asarray(y_train), minlength=2)
        test_counts = np.bincount(np.asarray(y_test), minlength=2)

        result = {
            "held_out_device_id": held_out_device_id,
            "held_out_device_name": device_names[held_out_device_id],
            "selector_train_devices": train_device_string,
            "classifier_train_devices": train_device_string,
            "held_out_excluded_from_selector": True,
            "held_out_excluded_from_classifier": True,
            "train_rows": train_rows,
            "test_rows": test_rows,
            "train_benign": int(train_counts[0]),
            "train_attack": int(train_counts[1]),
            "test_benign": int(test_counts[0]),
            "test_attack": int(test_counts[1]),
            "selector_feature_count": 115,
            "classifier_feature_count": TOP_N,
            **metrics,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "selector_load_seconds": selector_load_seconds,
            "selector_fit_seconds": selector_fit_seconds,
            "top20_projection_seconds": projection_seconds,
            "final_fit_seconds": final_fit_seconds,
            "test_load_seconds": test_load_seconds,
            "predict_seconds": predict_seconds,
            "top20_ranked_features": " | ".join(ranked_top20),
            "model_feature_order": " | ".join(model_feature_columns),
            "feature_importance_path": relative_output_path(paths["feature_importance"]),
            "top20_path": relative_output_path(paths["top20"]),
            "metrics_path": relative_output_path(paths["metrics"]),
            "confusion_matrix_path": relative_output_path(paths["confusion_matrix"]),
            "classification_report_path": relative_output_path(
                paths["classification_report"]
            ),
        }

        matrix_frame = pd.DataFrame(
            matrix,
            index=["actual_benign", "actual_attack"],
            columns=["predicted_benign", "predicted_attack"],
        )
        atomic_write_csv(pd.DataFrame([result]), paths["metrics"])
        atomic_write_csv(matrix_frame, paths["confusion_matrix"], index=True)
        atomic_write_text(report, paths["classification_report"])

        nested_lodo_results.append(result)
        nested_lodo_confusion_matrices[held_out_device_id] = matrix
        all_fold_top20_rows.extend(fold_top20.to_dict(orient="records"))

        # Refresh aggregate progress files after every completed fold.
        atomic_write_csv(pd.DataFrame(nested_lodo_results), ALL_RESULTS_PATH)
        atomic_write_csv(pd.DataFrame(all_fold_top20_rows), ALL_TOP20_PATH)

        print(
            f"  accuracy={metrics['accuracy']:.6f}, "
            f"attack_recall={metrics['attack_recall']:.6f}, "
            f"FP={fp:,}, FN={fn:,}"
        )
        print(f"  Top-20: {', '.join(ranked_top20)}")

        del (
            final_model,
            predictions,
            X_train_top20,
            X_test,
            y_train,
            y_test,
        )
        gc.collect()

nested_lodo_results_df = pd.DataFrame(nested_lodo_results).sort_values(
    "held_out_device_id"
).reset_index(drop=True)
all_folds_top20_df = pd.DataFrame(all_fold_top20_rows).sort_values(
    ["held_out_device_id", "rank"]
).reset_index(drop=True)
atomic_write_csv(nested_lodo_results_df, ALL_RESULTS_PATH)
atomic_write_csv(all_folds_top20_df, ALL_TOP20_PATH)

## 8. Fold-specific Top-20 lists

The 20×9 table makes fold-to-fold selector differences visible. Each column is one held-out device, and each row is an importance rank. The underlying long-form table, including importance values and actual model-column positions, is saved to `all_folds_top20.csv` and to nine separate per-device files.

In [ ]:
top20_by_fold = all_folds_top20_df.pivot(
    index="rank",
    columns="held_out_device_id",
    values="feature",
)
top20_by_fold.columns = [f"Held out device {device_id}" for device_id in top20_by_fold.columns]
display(top20_by_fold)

top20_frequency = (
    all_folds_top20_df.groupby("feature")
    .agg(
        folds_selected=("held_out_device_id", "nunique"),
        mean_rank=("rank", "mean"),
        best_rank=("rank", "min"),
        worst_rank=("rank", "max"),
    )
    .sort_values(["folds_selected", "mean_rank"], ascending=[False, True])
)
display(top20_frequency)

## 9. Per-device performance and aggregate statistics

In [ ]:
performance_columns = [
    "held_out_device_id",
    "held_out_device_name",
    "train_rows",
    "test_rows",
    "accuracy",
    "attack_precision",
    "attack_recall",
    "attack_f1",
    "benign_precision",
    "benign_recall",
    "benign_f1",
    "macro_f1",
    "tn",
    "fp",
    "fn",
    "tp",
    "selector_fit_seconds",
    "final_fit_seconds",
    "predict_seconds",
]
display(nested_lodo_results_df[performance_columns])

aggregate_metric_columns = [
    "accuracy",
    "attack_precision",
    "attack_recall",
    "attack_f1",
    "benign_precision",
    "benign_recall",
    "benign_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
]
aggregate_statistics = (
    nested_lodo_results_df[aggregate_metric_columns]
    .agg(["mean", "std", "min", "max"])
    .T
)
aggregate_statistics.columns = ["Mean", "Sample SD", "Minimum", "Maximum"]
display(aggregate_statistics)

## 10. Per-device metric visualization

In [ ]:
plot_metrics = {
    "accuracy": "Accuracy",
    "attack_precision": "Attack Precision",
    "attack_recall": "Attack Recall",
    "attack_f1": "Attack F1",
}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for axis, (column, title) in zip(axes.flat, plot_metrics.items()):
    sns.barplot(
        data=nested_lodo_results_df,
        x="held_out_device_id",
        y=column,
        color="#356aa0",
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel("Held-out device ID")
    axis.set_ylabel(title)
    axis.set_ylim(
        max(0.0, nested_lodo_results_df[column].min() - 0.01),
        1.0001,
    )
plt.show()

## 11. Artifact and isolation validation

These checks confirm nine complete folds, 20 unique fold-specific features, identical selector/classifier training-device sets, held-out exclusion, valid confusion-matrix totals, and the required separate files. They validate the new nested output namespace only; legacy global-Top-20 LODO results are methodologically different and are not treated as equality references.

In [ ]:
assert len(nested_lodo_results_df) == 9
assert nested_lodo_results_df["held_out_device_id"].nunique() == 9
assert len(all_folds_top20_df) == 9 * TOP_N

validation_rows = []
for held_out_device_id in device_ids:
    result = nested_lodo_results_df.loc[
        nested_lodo_results_df["held_out_device_id"] == held_out_device_id
    ].iloc[0]
    fold_features = all_folds_top20_df.loc[
        all_folds_top20_df["held_out_device_id"] == held_out_device_id,
        "feature",
    ]
    paths = fold_output_paths(held_out_device_id)
    matrix_total = int(result[["tn", "fp", "fn", "tp"]].sum())
    train_ids = [int(value) for value in result["selector_train_devices"].split()]

    validation_rows.append(
        {
            "Device": held_out_device_id,
            "20 unique features": (
                len(fold_features) == TOP_N and fold_features.nunique() == TOP_N
            ),
            "Features subset of canonical 115": set(fold_features).issubset(
                canonical_feature_columns
            ),
            "Held out excluded from selector": held_out_device_id not in train_ids,
            "Same selector/classifier devices": (
                result["selector_train_devices"]
                == result["classifier_train_devices"]
            ),
            "Confusion total equals test rows": matrix_total == int(result["test_rows"]),
            "Feature-importance file exists": paths["feature_importance"].is_file(),
            "Top-20 file exists": paths["top20"].is_file(),
            "Performance file exists": paths["metrics"].is_file(),
            "Confusion file exists": paths["confusion_matrix"].is_file(),
            "Classification report exists": paths["classification_report"].is_file(),
        }
    )

artifact_validation = pd.DataFrame(validation_rows)
display(artifact_validation)
if not artifact_validation.drop(columns="Device").all().all():
    raise AssertionError("One or more nested LODO isolation/artifact checks failed.")

print(f"Aggregate performance: {ALL_RESULTS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Aggregate Top-20 lists: {ALL_TOP20_PATH.relative_to(PROJECT_ROOT)}")

## 12. Reproducibility summary

- **Outer validation:** nine LODO folds; one complete device held out per fold.
- **Feature selection:** a fresh 115-feature Random Forest fitted only on the eight training devices in that fold.
- **Final classifier:** a second fresh Random Forest fitted on the same eight devices using that fold's Top-20 set.
- **Model configuration:** `n_estimators=100`, `random_state=42`, `n_jobs=-1`, `verbose=1`; remaining parameters use scikit-learn defaults.
- **Labels:** `binary_target=0` is benign and `binary_target=1` is attack; attack is the positive class.
- **Scaling/balancing:** none.
- **Column ordering:** importance lists are saved by rank; classifier matrices use canonical source-header order restricted to the selected set.
- **Persistence:** every fold is saved separately under `outputs/reports/lodo_nested_top20/`, with aggregate CSVs refreshed after every completed fold.

This design removes the earlier held-out-device feature-selection leakage: no held-out feature values or labels participate in either fitted Random Forest.